# Headroom API exploration

Hands-on tour of every documented API surface in [`headroom-ai`](https://github.com/chopratejas/headroom) **v0.27.0** (PyPI `headroom-ai[all]`), with sample data and visible output at every step.

Built from ground truth against the installed package — the upstream docs at [headroom-docs.vercel.app/docs/api-reference](https://headroom-docs.vercel.app/docs/api-reference) drift from the actual API in places (e.g. `generate_report` no longer takes `format=`/`period=`, `SmartCrusher.crush()` now takes a JSON string, `score_items()` was renamed to `score()` / `score_batch()`).

Structure (12 sections, ~30 code cells):

1. `headroom.compress()` — the main entry point
2. `HeadroomClient` — drop-in SDK wrapper (`chat.completions.simulate`, `validate_setup`, `get_stats`, `get_summary`, `get_metrics`)
3. Result types — `CompressResult`, `SimulationResult`, `WasteSignals`, `RequestMetrics`
4. Configuration — `HeadroomConfig`, `SmartCrusherConfig`, `CacheAlignerConfig`, `CompressionProfile`, `PROFILE_PRESETS`, `ScoringWeights`, `RelevanceScorerConfig`
5. Transforms — `SmartCrusher.crush`, `CacheAligner.apply`, `detect_volatile_content`, `TransformPipeline`
6. Providers — `AnthropicProvider`, `OpenAIProvider`, `GoogleProvider`
7. Relevance scoring — `BM25Scorer`, `EmbeddingScorer`, `HybridScorer`, `create_scorer`, `embedding_available`
8. Tokenization — `Tokenizer`, `count_tokens_text`, `count_tokens_messages`
9. Errors — `HeadroomError` hierarchy and `.details`
10. Reporting — `generate_report`
11. CLI — `proxy`, `wrap`, `evals`, `learn`
12. Where to go next

## 0. Setup

```bash
uv sync
test -f .env || cat > .env <<'EOF'
MINIMAX_API_KEY=sk-cp-your-key-here
EOF
```

All API cells below run **without** a live API key — they only exercise in-process transforms, scorers, and result shapes. Only the `HeadroomClient` cell needs `.env` (it constructs a real Anthropic client to wrap). Skip that cell if you don't have one.

In [1]:
import json
from pprint import pprint
import headroom
print("headroom.__version__:", headroom.__version__)
print("location        :", headroom.__file__)

headroom.__version__: 0.27.0
location        : /Users/shreyassk/VSCode/headroom/.venv/lib/python3.11/site-packages/headroom/__init__.py


## 1. `headroom.compress()` — the main entry point

Signatures vary a bit across versions; v0.27 exposes `compress(messages, *, model, ...)` and returns a `CompressResult` with snake_case attrs (the docs' camelCase names exist too as compatibility aliases on some fields). The default mode is **token-saving** — runs the full `TransformPipeline` (SmartCrusher + CacheAligner + content router) end-to-end on the messages list.

In [2]:
import json
from headroom import compress

# Synthetic search-result payload in a tool_result block — that's where
# SmartCrusher crushes hardest. protect_user_messages=True (default)
# leaves user-turn content alone.
messages = [
    {"role": "system",  "content": "You are a search assistant. Reply concisely."},
    {"role": "user",    "content": "Find Python tutorials."},
    {"role": "assistant","content": None, "tool_calls": [{
        "id": "call_1", "type": "function",
        "function": {"name": "search", "arguments": '{"q": "python"}'},
    }]},
    {"role": "tool",    "tool_call_id": "call_1",
     "content": json.dumps({
         "results": [
             {"title": f"Result {i}",
              "snippet": "description padding " * 5,
              "score": 100 - i,
              "url": f"https://example.com/r/{i}"}
             for i in range(200)
         ]
     })},
    {"role": "user",    "content": "What are the top 3 results?"},
]

result = compress(messages, model="claude-sonnet-4-6")

print(f"type                : {type(result).__name__}")
print(f"tokens_before       : {result.tokens_before}")
print(f"tokens_after        : {result.tokens_after}")
print(f"tokens_saved        : {result.tokens_saved}")
print(f"compression_ratio   : {result.compression_ratio:.3f}")
print(f"transforms_applied  : {result.transforms_applied}")
print(f"messages length     : {len(result.messages)}  (original: {len(messages)})")
print(f"savings             : {(result.tokens_before - result.tokens_after) / result.tokens_before * 100:.1f}%")


type                : CompressResult
tokens_before       : 10834
tokens_after        : 8165
tokens_saved        : 2669
compression_ratio   : 0.246
transforms_applied  : ['router:protected:user_message', 'router:protected:user_message', 'router:smart_crusher:0.58']
messages length     : 5  (original: 5)
savings             : 24.6%


`compress()` is the same code path the proxy runs in-process. The `transforms_applied` list names each transform that ran (e.g. `router:smart_crusher:0.58` — the trailing number is the achieved compression ratio for that stage). The 0.27 `CompressResult` schema is intentionally small: `messages`, `tokens_before`, `tokens_after`, `tokens_saved`, `compression_ratio`, `transforms_applied`. The upstream docs mention `ccr_hashes` / `compressed` fields that don't exist in this version.


## 2. `HeadroomClient` — SDK wrapper

Wraps your existing `anthropic.Anthropic` / `openai.OpenAI` client and exposes `chat.completions.simulate()` (no API call — returns a `SimulationResult`), plus `validate_setup()`, `get_stats()`, `get_summary()`, `get_metrics()`.

In [3]:
import os, json
from dotenv import load_dotenv
load_dotenv()  # picks up MINIMAX_API_KEY from .env

import anthropic
from headroom import HeadroomClient
from headroom.providers import AnthropicProvider

mini = anthropic.Anthropic(
    base_url="https://api.minimax.io/anthropic",
    api_key=os.environ["MINIMAX_API_KEY"],
)
provider = AnthropicProvider(client=mini, warn=False)
hr = HeadroomClient(
    original_client=mini,
    provider=provider,
    default_mode="audit",
    enable_cache_optimizer=False,
    enable_semantic_cache=False,
)
print("client ready, provider:", provider.name)


client ready, provider: anthropic


In [4]:
# 2a. validate_setup() — sanity-check provider / storage / config wiring
print(json.dumps(hr.validate_setup(), indent=2, default=str))

{
  "valid": true,
  "provider": {
    "ok": true,
    "name": "anthropic",
    "error": null
  },
  "storage": {
    "ok": true,
    "url": "sqlite:////var/folders/_r/nhh22c_d1db8kbw1ft0xmlwc0000gn/T/headroom.db",
    "error": null
  },
  "config": {
    "ok": true,
    "mode": "audit",
    "error": null
  },
  "cache_optimizer": {
    "ok": true,
    "name": null,
    "error": null
  }
}


In [5]:
# 2b. chat.completions.simulate() — dry-run compression.
#
# MiniMax rejects /count_tokens on synthetic tool_use/tool_result
# pairs with HTTP 400; Headroom falls back to EstimatingTokenCounter.
# Wrap in warnings.catch_warnings() to silence the UserWarning.
import warnings

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    sim = hr.chat.completions.simulate(model="MiniMax-M3", messages=messages)

print("type                   :", type(sim).__name__)
print("tokens_before          :", sim.tokens_before)
print("tokens_after           :", sim.tokens_after)
print("tokens_saved           :", sim.tokens_saved)
print("estimated_savings      :", sim.estimated_savings, "(str in 0.27)")
print("transforms             :", sim.transforms)
print("messages_optimized len :", len(sim.messages_optimized))
print("stable_prefix_hash     :", sim.stable_prefix_hash)
print("cache_alignment_score  :", sim.cache_alignment_score)
print("block_breakdown        :")
pprint(sim.block_breakdown)
print("waste_signals          :")
pprint(sim.waste_signals)


Unknown Anthropic model 'MiniMax-M3': unknown provider, using conservative default (128,000 tokens). To configure explicitly, set HEADROOM_MODEL_LIMITS env var or add to ~/.headroom/models.json


type                   : SimulationResult
tokens_before          : 9078
tokens_after           : 5682
tokens_saved           : 3396
estimated_savings      : N/A per request (str in 0.27)
transforms             : ['router:protected:system_message', 'router:protected:user_message', 'router:protected:user_message', 'router:smart_crusher:0.58']
messages_optimized len : 5
stable_prefix_hash     : 9a8fe4d7ed9b199b
cache_alignment_score  : 100.0
block_breakdown        :
{'system': 16, 'tool_call': 17, 'tool_result': 9028, 'user': 20}
waste_signals          :
{'base64': 0,
 'dynamic_date': 0,
 'html_noise': 0,
 'json_bloat': 9024,
 'repetition': 0,
 'reread': 0,
 'reread_compressed': 0,
 'whitespace': 0}


In [6]:
# 2c. get_stats() / get_summary() / get_metrics() — session-level counters.
#     Empty so far (no real requests made); the shape is what matters.
print("=== get_stats ===")
pprint(hr.get_stats())
print()
print("=== get_summary ===")
pprint(hr.get_summary())
print()
print("=== get_metrics ===")
print(hr.get_metrics())

=== get_stats ===
{'config': {'cache_optimizer': None,
            'mode': 'audit',
            'provider': 'anthropic',
            'semantic_cache': False},
 'session': {'cache_hits': 0,
             'requests_audit': 0,
             'requests_optimized': 0,
             'requests_total': 0,
             'tokens_saved_total': 0},
 'transforms': {'cache_aligner_enabled': False, 'smart_crusher_enabled': True}}

=== get_summary ===
{'audit_count': 0,
 'avg_cache_alignment': 0,
 'avg_tokens_saved': 0,
 'optimize_count': 0,
 'total_requests': 0,
 'total_tokens_after': 0,
 'total_tokens_before': 0,
 'total_tokens_saved': 0}

=== get_metrics ===
[]


## 3. Result types

Dataclasses returned by the compression / scoring pipeline.

In [7]:
from headroom import CompressResult, SimulationResult, RequestMetrics, WasteSignals, TransformResult, CachePrefixMetrics

print("=== CompressResult (return of headroom.compress) ===")
print(" fields :", [f.name for f in CompressResult.__dataclass_fields__.values()])
print()
print("=== SimulationResult (return of chat.completions.simulate) ===")
print(" fields :", [f.name for f in SimulationResult.__dataclass_fields__.values()])
print()
print("=== WasteSignals ===")
print(" fields :", [f.name for f in WasteSignals.__dataclass_fields__.values()])
print()
print("=== RequestMetrics ===")
print(" fields :", [f.name for f in RequestMetrics.__dataclass_fields__.values()])
print()
print("=== TransformResult (single transform's output) ===")
print(" fields :", [f.name for f in TransformResult.__dataclass_fields__.values()])
print()
print("=== CachePrefixMetrics ===")
print(" fields :", [f.name for f in CachePrefixMetrics.__dataclass_fields__.values()])

=== CompressResult (return of headroom.compress) ===
 fields : ['messages', 'tokens_before', 'tokens_after', 'tokens_saved', 'compression_ratio', 'transforms_applied']

=== SimulationResult (return of chat.completions.simulate) ===
 fields : ['tokens_before', 'tokens_after', 'tokens_saved', 'transforms', 'estimated_savings', 'messages_optimized', 'block_breakdown', 'waste_signals', 'stable_prefix_hash', 'cache_alignment_score']

=== WasteSignals ===
 fields : ['json_bloat_tokens', 'html_noise_tokens', 'base64_tokens', 'whitespace_tokens', 'dynamic_date_tokens', 'repetition_tokens', 'reread_tokens', 'reread_compressed_tokens']

=== RequestMetrics ===
 fields : ['request_id', 'timestamp', 'model', 'stream', 'mode', 'tokens_input_before', 'tokens_input_after', 'tokens_output', 'block_breakdown', 'waste_signals', 'stable_prefix_hash', 'cache_alignment_score', 'cached_tokens', 'cache_optimizer_used', 'cache_optimizer_strategy', 'cacheable_tokens', 'breakpoints_inserted', 'estimated_cache_hi

## 4. Configuration

Every knob exposed in the Python API. The headline change in 0.27 is the move from bespoke `RollingWindowConfig` / `IntelligentContextConfig` / `ScoringWeights` classes to a unified `CompressionProfile` + `PROFILE_PRESETS` table keyed per tool.

In [8]:
from headroom import (
    HeadroomConfig, HeadroomMode,
    SmartCrusherConfig, CacheAlignerConfig,
    RelevanceScorerConfig,
)
from headroom.config import CompressionProfile, PROFILE_PRESETS, DEFAULT_TOOL_PROFILES, DEFAULT_MODEL_CONTEXT_LIMITS

print("=== PROFILE_PRESETS ===")
for name, profile in PROFILE_PRESETS.items():
    print(f"  {name:13s} -> {profile}")
print()
print("=== DEFAULT_TOOL_PROFILES (per-tool default) ===")
pprint(DEFAULT_TOOL_PROFILES)
print()
custom = CompressionProfile(bias=0.5, min_k=2, max_k=20)
print("custom profile :", custom)
print()
print("=== DEFAULT_MODEL_CONTEXT_LIMITS (sampled) ===")
for k in list(DEFAULT_MODEL_CONTEXT_LIMITS)[:5]:
    print(f"  {k:40s} -> {DEFAULT_MODEL_CONTEXT_LIMITS[k]:,}")


=== PROFILE_PRESETS ===
  conservative  -> CompressionProfile(bias=1.5, min_k=5, max_k=None)
  moderate      -> CompressionProfile(bias=1.0, min_k=3, max_k=None)
  aggressive    -> CompressionProfile(bias=0.7, min_k=3, max_k=None)

=== DEFAULT_TOOL_PROFILES (per-tool default) ===
{'Bash': CompressionProfile(bias=1.0, min_k=3, max_k=None),
 'Grep': CompressionProfile(bias=1.5, min_k=5, max_k=None),
 'WebFetch': CompressionProfile(bias=0.7, min_k=3, max_k=None),
 'bash': CompressionProfile(bias=1.0, min_k=3, max_k=None),
 'grep': CompressionProfile(bias=1.5, min_k=5, max_k=None),
 'webfetch': CompressionProfile(bias=0.7, min_k=3, max_k=None)}

custom profile : CompressionProfile(bias=0.5, min_k=2, max_k=20)

=== DEFAULT_MODEL_CONTEXT_LIMITS (sampled) ===


In [9]:
# 4b. HeadroomConfig — top-level config tree; sub-configs are attributes.
cfg = HeadroomConfig()
print("default_mode          :", cfg.default_mode)
print("output_buffer_tokens  :", cfg.output_buffer_tokens)
print("smart_crusher         :", cfg.smart_crusher)
print("cache_aligner         :", cfg.cache_aligner)
print("ccr                   :", cfg.ccr)
print("prefix_freeze         :", cfg.prefix_freeze)
print("cache_optimizer       :", cfg.cache_optimizer)
print("store_url             :", cfg.store_url)
print()
print("HeadroomMode members  :", [m.value for m in HeadroomMode])

default_mode          : HeadroomMode.AUDIT
output_buffer_tokens  : 4000
smart_crusher         : SmartCrusherConfig(enabled=True, min_items_to_analyze=5, min_tokens_to_crush=200, variance_threshold=2.0, uniqueness_threshold=0.1, similarity_threshold=0.8, max_items_after_crush=15, preserve_change_points=True, factor_out_constants=False, include_summaries=False, use_feedback_hints=True, toin_confidence_threshold=0.3, relevance=RelevanceScorerConfig(tier='hybrid', bm25_k1=1.5, bm25_b=0.75, embedding_model='all-MiniLM-L6-v2', hybrid_alpha=0.5, adaptive_alpha=True, relevance_threshold=0.25), anchor=AnchorConfig(anchor_budget_pct=0.25, min_anchor_slots=3, max_anchor_slots=12, default_front_weight=0.5, default_back_weight=0.4, default_middle_weight=0.1, search_front_weight=0.75, search_back_weight=0.15, logs_front_weight=0.15, logs_back_weight=0.75, recency_keywords=('latest', 'recent', 'last', 'newest', 'current', 'now'), historical_keywords=('first', 'oldest', 'earliest', 'original', 'initia

In [10]:
# 4c-4e. Config attribute introspection — robust against schema drift.
from headroom import SmartCrusherConfig, CacheAlignerConfig, RelevanceScorerConfig

def public_attrs(obj):
    return {k: getattr(obj, k) for k in dir(obj)
            if not k.startswith('_') and not callable(getattr(obj, k))}

def show(label, obj):
    print(f"=== {label} ===")
    for k, v in public_attrs(obj).items():
        if hasattr(v, '__dict__') or isinstance(v, type):
            print(f"  {k:30s} -> {type(v).__name__}")
        else:
            print(f"  {k:30s} = {repr(v)[:80]}")
    print()

show("SmartCrusherConfig",  SmartCrusherConfig())
show("CacheAlignerConfig",  CacheAlignerConfig())
show("RelevanceScorerConfig", RelevanceScorerConfig())


=== SmartCrusherConfig ===
  anchor                         -> AnchorConfig
  compaction_core_field_fraction = 0.8
  compaction_heterogeneous_core_ratio = 0.6
  compaction_max_buckets         = 8
  compaction_max_flatten_inner_keys = 6
  compaction_min_buckets         = 2
  dedup_identical_items          = True
  enabled                        = True
  factor_out_constants           = False
  first_fraction                 = 0.3
  include_summaries              = False
  last_fraction                  = 0.15
  lossless_min_savings_ratio     = 0.15
  max_items_after_crush          = 15
  min_items_to_analyze           = 5
  min_tokens_to_crush            = 200
  preserve_change_points         = True
  relevance                      -> RelevanceScorerConfig
  similarity_threshold           = 0.8
  toin_confidence_threshold      = 0.3
  uniqueness_threshold           = 0.1
  use_feedback_hints             = True
  variance_threshold             = 2.0

=== CacheAlignerConfig ===
  collapse

## 5. Transforms — direct use

Every transform that runs inside `compress()` is also usable standalone. SmartCrusher is the heavy lifter; CacheAligner is detector-only (never mutates the system prompt — see `setting_up.ipynb` §3 for why).

In [11]:
# 5a. SmartCrusher.crush() — direct call. NOTE: takes a JSON STRING, not a list.
import json
from headroom import SmartCrusher, SmartCrusherConfig

rows = [
    {"id": i, "name": f"item {i}", "desc": "padding " * 6, "score": i * 0.1}
    for i in range(50)
]
crusher = SmartCrusher(SmartCrusherConfig())
r = crusher.crush(json.dumps(rows), query="python")

print(f"type           : {type(r).__name__}")
print(f"strategy       : {r.strategy}")
print(f"was_modified   : {r.was_modified}")
print(f"original bytes : {len(r.original)}")
print(f"compressed bytes: {len(r.compressed)}")
print(f"savings        : {(1 - len(r.compressed) / len(r.original)) * 100:.1f}%")
print()
print("=== first 200 chars of compressed ===")
print(r.compressed[:200])

type           : CrushResult
strategy       : lossless:table(50->len=3499)
was_modified   : True
original bytes : 5500
compressed bytes: 3552
savings        : 35.4%

=== first 200 chars of compressed ===
"[50]{desc:string,id:int,name:string,score:float}\npadding padding padding padding padding padding ,0,item 0,0.0\npadding padding padding padding padding padding ,1,item 1,0.1\npadding padding padding


In [12]:
# 5b. CacheAligner.apply() — detector-only. Returns TransformResult with
#     messages unchanged plus a stable_prefix_hash for cache-hit tracking.
from headroom import CacheAligner
from headroom.transforms.cache_aligner import detect_volatile_content
from headroom.tokenizer import Tokenizer
from headroom.tokenizers import EstimatingTokenCounter

aligner = CacheAligner()
tok = Tokenizer(EstimatingTokenCounter())

stable = [
    {"role": "system", "content": "You are a concise assistant. Reply in one sentence."},
    {"role": "user",   "content": "What is 2 + 2?"},
]
r = aligner.apply(stable, tok)
print("=== stable prompt ===")
print(f"  messages unchanged : {r.messages == stable}")
print(f"  transforms_applied: {r.transforms_applied}")
print(f"  warnings          : {r.warnings}")
print(f"  markers_inserted  : {r.markers_inserted}")
print(f"  cache_metrics     : {r.cache_metrics}")
print(f"  alignment_score   : {aligner.get_alignment_score(stable):.1f}%")

print()
unstable = [
    {"role": "system", "content": (
        "You are assistant. Session: 550e8400-e29b-41d4-a716-446655440000. "
        "Started at 2026-06-24T12:00:00Z. "
        "Token: eyJhbGciOiJIUzI1NiJ9.eyJzdWIiOiIxMjM0NTY3ODkwIn0.SflKxw."
    )},
    {"role": "user", "content": "What is 2 + 2?"},
]
r2 = aligner.apply(unstable, tok)
print("=== unstable prompt (UUID + ISO timestamp + JWT) ===")
print(f"  warnings          : {r2.warnings}")
print(f"  cache_metrics     : {r2.cache_metrics}")
print(f"  alignment_score   : {aligner.get_alignment_score(unstable):.1f}%")

print()
print("=== detect_volatile_content() findings ===")
for f in detect_volatile_content(unstable[0]["content"]):
    print(f"  {f.label:9s} sample={f.sample}")

CacheAligner: detected volatile content in system prompt (iso8601=1, jwt=1, uuid=1); cache prefix unstable. Move dynamic values out of the system prompt to recover cache hits.


=== stable prompt ===
  messages unchanged : True
  transforms_applied: []
  warnings          : []
  markers_inserted  : ['stable_prefix_hash:33c44988517116bf']
  cache_metrics     : CachePrefixMetrics(stable_prefix_bytes=51, stable_prefix_tokens_est=13, stable_prefix_hash='33c44988517116bf', prefix_changed=False, previous_hash=None)
  alignment_score   : 100.0%

=== unstable prompt (UUID + ISO timestamp + JWT) ===
  warnings          : ['CacheAligner: detected volatile content in system prompt (iso8601=1, jwt=1, uuid=1); cache prefix unstable. Move dynamic values out of the system prompt to recover cache hits.']
  cache_metrics     : CachePrefixMetrics(stable_prefix_bytes=162, stable_prefix_tokens_est=43, stable_prefix_hash='b84b437335fc7d1a', prefix_changed=True, previous_hash='33c44988517116bf')
  alignment_score   : 70.0%

=== detect_volatile_content() findings ===
  uuid      sample=550e8400...0000
  iso8601   sample=2026-06-...:00Z
  jwt       sample=eyJhbGci...lKxw


In [13]:
# 5c. TransformPipeline — runs each transform in order.
from headroom import TransformPipeline
from headroom.transforms.cache_aligner import CacheAligner
from headroom.transforms.smart_crusher import SmartCrusher

pipeline = TransformPipeline(transforms=[
    CacheAligner(),
    SmartCrusher(SmartCrusherConfig()),
])
out = pipeline.apply(messages, model="claude-sonnet-4-6", model_limit=200_000)
print("type              :", type(out).__name__)
print("messages_in       :", len(messages))
print("messages_out      :", len(out.messages))
print("tokens_before     :", out.tokens_before)
print("tokens_after      :", out.tokens_after)
print("transforms_applied:", out.transforms_applied)
print()
sim = pipeline.simulate(messages, model="claude-sonnet-4-6", model_limit=200_000)
print("simulate()        :", "before:", sim.tokens_before, "after:", sim.tokens_after)


type              : TransformResult
messages_in       : 5
messages_out      : 5
tokens_before     : 10834
tokens_after      : 8179
transforms_applied: ['smart_crush:1:search', 'smart:lossless:table(200->len=28117)']

simulate()        : before: 10834 after: 8179


## 6. Providers

Provider wrappers feed `HeadroomClient` with model-aware token counting, cost estimation, and provider-specific optimizations (Anthropic prompt cache control, OpenAI prefix caching, Google context caching).

In [14]:
import inspect
from headroom import AnthropicProvider, OpenAIProvider
from headroom.providers import GoogleProvider, CohereProvider

for cls in [AnthropicProvider, OpenAIProvider, GoogleProvider, CohereProvider]:
    methods = [m for m in dir(cls) if not m.startswith('_') and callable(getattr(cls, m))]
    print(f"=== {cls.__name__} ===")
    print(f"  ctor: {inspect.signature(cls)}")
    print(f"  methods: {methods}")
    print()


=== AnthropicProvider ===
  ctor: (client: Any = None, context_limits: dict[str, int] | None = None, warn: bool = True)
  methods: ['estimate_cost', 'get_context_limit', 'get_output_buffer', 'get_token_counter', 'supports_model']

=== OpenAIProvider ===
  ctor: (context_limits: 'dict[str, int] | None' = None)
  methods: ['estimate_cost', 'get_context_limit', 'get_output_buffer', 'get_token_counter', 'supports_model']

=== GoogleProvider ===
  ctor: (client: 'Any' = None)
  methods: ['estimate_cost', 'get_context_limit', 'get_openai_compatible_url', 'get_output_buffer', 'get_token_counter', 'supports_model']

=== CohereProvider ===
  ctor: (client: 'Any' = None)
  methods: ['estimate_cost', 'get_context_limit', 'get_output_buffer', 'get_token_counter', 'supports_model']



In [24]:
# Trigger the venv patch that pre-registers MiniMax-M3 in litellm.model_cost.
# HeadroomClient's import chain doesn't pull in headroom.pricing.litellm_pricing,
# so we have to do it explicitly to make `litellm.model_cost['MiniMax-M3']` resolve.
from headroom.pricing import litellm_pricing  # noqa: F401  — runs _register_minimax_pricing()

# AnthropicProvider in action (uses MiniMax-M3 as a stand-in for Claude).
from headroom import AnthropicProvider
ap = AnthropicProvider(client=mini, warn=False)

print("provider.name             :", ap.name)
print("supports_model(MiniMax-M3):", ap.supports_model("MiniMax-M3"))
print("context_limit(MiniMax-M3) :", ap.get_context_limit("MiniMax-M3"))
print("output_buffer(MiniMax-M3) :", ap.get_output_buffer("MiniMax-M3"))
counter = ap.get_token_counter("MiniMax-M3")
print("counter class             :", type(counter).__name__)

# AnthropicProvider.estimate_cost returns None for MiniMax-M3 because of an
# upstream LiteLLM API drift: completion_cost() no longer accepts
# prompt_tokens=/completion_tokens= kwargs in this LiteLLM version, so the
# call raises and the provider falls back to its hard-coded manual pricing
# (which has no MiniMax entry). Compare with claude-haiku-4-5, which works.
print("AnthropicProvider.estimate_cost(MiniMax-M3, 1k in, 100 out):",
      ap.estimate_cost(input_tokens=1000, output_tokens=100, model="MiniMax-M3"))
print("  (compare: claude-haiku-4-5 returns)",
      ap.estimate_cost(input_tokens=1_000_000, output_tokens=100_000, model="claude-haiku-4-5"))

# --- Workaround: call LiteLLM directly with the prefixed key ---
print()
print("=== LiteLLM direct lookup (works because LiteLLM ships a `minimax/` provider) ===")

import litellm
from litellm.types.utils import ModelResponse, Usage

# 1. cost_per_token() — old-but-still-supported API, takes the prefixed key.
#    The bare name 'MiniMax-M3' fails because LiteLLM can't route it to a provider.
input_cost, output_cost = litellm.cost_per_token(
    model="minimax/MiniMax-M3", prompt_tokens=1000, completion_tokens=100,
)
print(f"litellm.cost_per_token('minimax/MiniMax-M3', 1k, 100) = ({input_cost}, {output_cost})")
print(f"  total: ${input_cost + output_cost:.4f}")

# 2. completion_cost() — modern API; pass a ModelResponse with the prefixed model.
resp = ModelResponse(
    id="x", choices=[], model="minimax/MiniMax-M3",
    usage=Usage(prompt_tokens=1000, completion_tokens=100, total_tokens=1100),
)
print(f"litellm.completion_cost(ModelResponse('minimax/MiniMax-M3', 1k, 100)) =",
      litellm.completion_cost(completion_response=resp))

# 3. Direct dict lookup from litellm.model_cost["MiniMax-M3"] (the bare-key
#    entry exists thanks to the venv patch in headroom/pricing/litellm_pricing.py).
entry = litellm.model_cost["MiniMax-M3"]
print()
print("litellm.model_cost['MiniMax-M3'] (bare-key, populated by venv patch):")
for k in ["input_cost_per_token", "output_cost_per_token", "cache_read_input_token_cost", "max_input_tokens"]:
    v = entry.get(k)
    if v is None:
        print(f"  {k:30s} = None")
    elif "per_token" in k:
        print(f"  {k:30s} = {v}  (${v * 1_000_000:.4f}/M)")
    else:
        print(f"  {k:30s} = {v}")

# 4. End-to-end: cost for 1M input + 100k output via the bare-key entry
in_cost  = 1_000_000 * entry["input_cost_per_token"]
out_cost = 100_000   * entry["output_cost_per_token"]
print()
print(f"1M input + 100k output via bare-key lookup = ${in_cost + out_cost:.4f}")

# 5. Cache-read cost (Anthropic prompt cache pricing)
cache_rate = entry.get("cache_read_input_token_cost", 0) or 0
print(f"cache_read rate = ${cache_rate * 1_000_000:.2f}/M")
print(f"1M cached-input tokens = ${1_000_000 * cache_rate:.4f}")


provider.name             : anthropic
supports_model(MiniMax-M3): False
context_limit(MiniMax-M3) : 1000000
output_buffer(MiniMax-M3) : 4000
counter class             : AnthropicTokenCounter
AnthropicProvider.estimate_cost(MiniMax-M3, 1k in, 100 out): None
  (compare: claude-haiku-4-5 returns) 1.2000000000000002

=== LiteLLM direct lookup (works because LiteLLM ships a `minimax/` provider) ===
litellm.cost_per_token('minimax/MiniMax-M3', 1k, 100) = (0.0003, 0.00011999999999999999)
  total: $0.0004
litellm.completion_cost(ModelResponse('minimax/MiniMax-M3', 1k, 100)) = 0.00041999999999999996

litellm.model_cost['MiniMax-M3'] (bare-key, populated by venv patch):
  input_cost_per_token           = 3e-07  ($0.3000/M)
  output_cost_per_token          = 1.2e-06  ($1.2000/M)
  cache_read_input_token_cost    = 6e-08
  max_input_tokens               = 1000000

1M input + 100k output via bare-key lookup = $0.4200
cache_read rate = $0.06/M
1M cached-input tokens = $0.0600


## 7. Relevance scoring

Headroom uses BM25 as the default; you can upgrade to embeddings or a hybrid. `create_scorer(tier)` is the factory — `tier` ∈ `{'bm25', 'embedding', 'hybrid'}`. The `Hybrid` scorer auto-tunes `alpha` when `adaptive=True`.

In [16]:
from headroom import BM25Scorer, EmbeddingScorer, HybridScorer, create_scorer, embedding_available

print("embedding_available()  :", embedding_available())
print()

# 7a. BM25 — keyword-based, no model download.
bm25 = BM25Scorer()
items = [
    "Python tutorial for beginners",
    "How to cook rice perfectly",
    "Advanced Python decorators and metaclasses",
    "JavaScript async/await patterns",
]
scores = bm25.score_batch(items, context="python programming")
print("=== BM25Scorer.score_batch ===")
for item, s in zip(items, scores):
    print(f"  {s.score:.3f}  matched={s.matched_terms}  | {item}")

embedding_available()  : True

=== BM25Scorer.score_batch ===
  0.073  matched=['python']  | Python tutorial for beginners
  0.000  matched=[]  | How to cook rice perfectly
  0.066  matched=['python']  | Advanced Python decorators and metaclasses
  0.000  matched=[]  | JavaScript async/await patterns


In [17]:
# 7b. Embedding — semantic; needs the headroom-ai[relevance] extra.
emb = EmbeddingScorer()
print("EmbeddingScorer.is_available():", emb.is_available())
scores = emb.score_batch(items, context="python programming")
print("=== EmbeddingScorer.score_batch ===")
for item, s in zip(items, scores):
    print(f"  {s.score:.3f}  reason={s.reason}  | {item}")

EmbeddingScorer.is_available(): True
=== EmbeddingScorer.score_batch ===
  0.818  reason=Embedding: 0.82  | Python tutorial for beginners
  0.499  reason=Embedding: 0.50  | How to cook rice perfectly
  0.695  reason=Embedding: 0.70  | Advanced Python decorators and metaclasses
  0.604  reason=Embedding: 0.60  | JavaScript async/await patterns


In [18]:
# 7c. Hybrid — blends BM25 + embeddings. alpha=1.0 = pure BM25, 0.0 = pure embedding.
hybrid = HybridScorer(alpha=0.5, adaptive=True)
scores = hybrid.score_batch(items, context="python programming")
print(f"alpha=0.5, adaptive={hybrid.adaptive}")
print("=== HybridScorer.score_batch ===")
for item, s in zip(items, scores):
    print(f"  {s.score:.3f}  reason={s.reason}  | {item}")

# 7d. create_scorer factory.
for tier in ["bm25", "hybrid"]:
    s = create_scorer(tier=tier)
    print(f"\ncreate_scorer(tier={tier!r}) -> {type(s).__name__}")

alpha=0.5, adaptive=True
=== HybridScorer.score_batch ===
  0.445  reason=Hybrid (α=0.50): BM25=0.07, Emb=0.82  | Python tutorial for beginners
  0.250  reason=Hybrid (α=0.50): BM25=0.00, Emb=0.50  | How to cook rice perfectly
  0.381  reason=Hybrid (α=0.50): BM25=0.07, Emb=0.70  | Advanced Python decorators and metaclasses
  0.302  reason=Hybrid (α=0.50): BM25=0.00, Emb=0.60  | JavaScript async/await patterns

create_scorer(tier='bm25') -> BM25Scorer

create_scorer(tier='hybrid') -> HybridScorer


## 8. Tokenization

Two equivalent paths: instantiate a `Tokenizer(model=...)` for repeated use, or call the module-level `count_tokens_text` / `count_tokens_messages` helpers.

In [19]:
from headroom import Tokenizer, count_tokens_text, count_tokens_messages
from headroom.tokenizers import EstimatingTokenCounter

counter = EstimatingTokenCounter()
tok = Tokenizer(token_counter=counter, model="claude-sonnet-4-6")

text = "The quick brown fox jumps over the lazy dog."
msgs = [
    {"role": "system", "content": "You are a concise assistant."},
    {"role": "user",   "content": text},
]

print("=== Tokenizer instance ===")
print(f"  count_text       : {tok.count_text(text)}")
print(f"  count_messages   : {tok.count_messages(msgs)}")
print()
print("=== module-level helpers (take a TokenCounter, not model=) ===")
print(f"  count_tokens_text    : {count_tokens_text(text, token_counter=counter)}")
print(f"  count_tokens_messages: {count_tokens_messages(msgs, token_counter=counter)}")


=== Tokenizer instance ===
  count_text       : 11
  count_messages   : 32

=== module-level helpers (take a TokenCounter, not model=) ===
  count_tokens_text    : 11
  count_tokens_messages: 32


## 9. Errors

All errors descend from `HeadroomError` and carry a `details` dict.

In [20]:
from headroom import (
    HeadroomError, ConfigurationError, ProviderError,
    StorageError, CompressionError, ValidationError,
    CacheError, TokenizationError, TransformError,
)

classes = [
    HeadroomError, ConfigurationError, ProviderError,
    StorageError, CompressionError, ValidationError,
    CacheError, TokenizationError, TransformError,
]
for cls in classes:
    bases = " -> ".join(c.__name__ for c in cls.__mro__ if c is not object)
    print(f"  {cls.__name__:20s} : {bases}")

# Construct one with details and inspect.
try:
    raise ConfigurationError("demo", details={"field": "store_url", "got": None})
except HeadroomError as e:
    print()
    print("=== caught ===")
    print(f"  type    : {type(e).__name__}")
    print(f"  str(e)  : {e}")
    print(f"  details : {e.details}")

  HeadroomError        : HeadroomError -> Exception -> BaseException
  ConfigurationError   : ConfigurationError -> HeadroomError -> Exception -> BaseException
  ProviderError        : ProviderError -> HeadroomError -> Exception -> BaseException
  StorageError         : StorageError -> HeadroomError -> Exception -> BaseException
  CompressionError     : CompressionError -> HeadroomError -> Exception -> BaseException
  ValidationError      : ValidationError -> HeadroomError -> Exception -> BaseException
  CacheError           : CacheError -> HeadroomError -> Exception -> BaseException
  TokenizationError    : TokenizationError -> HeadroomError -> Exception -> BaseException
  TransformError       : TransformError -> HeadroomError -> Exception -> BaseException

=== caught ===
  type    : ConfigurationError
  str(e)  : demo (field=store_url, got=None)
  details : {'field': 'store_url', 'got': None}


## 10. Reporting

`generate_report(store_url, output_path, start_time=None, end_time=None) -> str` — reads a SQLite-backed metrics store and writes an HTML report. The upstream docs mention `format=` / `period=` kwargs that don't exist in 0.27; pass time bounds directly.

In [21]:
import os, tempfile
from datetime import datetime, timedelta
from headroom import generate_report

db_path = os.path.join(tempfile.gettempdir(), "headroom_report_demo.db")
if os.path.exists(db_path):
    os.remove(db_path)
html_path = generate_report(
    store_url=f"sqlite:///{db_path}",
    output_path=os.path.join(tempfile.gettempdir(), "headroom_report_demo.html"),
    start_time=datetime.utcnow() - timedelta(days=7),
    end_time=datetime.utcnow(),
)
print("report path :", html_path)
print("exists      :", os.path.exists(html_path))
print("size (bytes):", os.path.getsize(html_path))
with open(html_path) as f:
    head = f.read(400)
print()
print("=== first 400 chars ===")
print(head)

report path : /var/folders/_r/nhh22c_d1db8kbw1ft0xmlwc0000gn/T/headroom_report_demo.html
exists      : True
size (bytes): 10142

=== first 400 chars ===

<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Headroom Report - 2026-06-24 15:36:01</title>
    <style>
        * {
            box-sizing: border-box;
            margin: 0;
            padding: 0;
        }
        body {
            font-family: -apple-system, BlinkMacSystemFont, 'Segoe U


## 11. CLI surface

Everything in this notebook is also exposed via the `headroom` CLI.

In [22]:
import subprocess
out = subprocess.run(["uv", "run", "headroom", "--help"], capture_output=True, text=True)
print(out.stdout)

Usage: headroom [OPTIONS] COMMAND [ARGS]...

  Headroom - The Context Optimization Layer for LLM Applications.

  Manage memories, run the optimization proxy, and analyze metrics.

  Examples:
      headroom proxy              Start the optimization proxy
      headroom memory list        List stored memories
      headroom memory stats       Show memory statistics
      headroom update             Update Headroom to the latest release

Options:
  -v, --version  Show the version and exit.
  -?, --help     Show this message and exit.

Commands:
  agent-savings   Render or verify Codex/Claude/Cursor token-savings...
  audit-reads     Audit Read-tool traffic for compression opportunities.
  capture         Capture and compare network traffic for Headroom...
  copilot-auth    Manage Headroom's GitHub Copilot OAuth token.
  diff            Run difftastic (structural diff).
  doctor          Check that the Headroom proxy and client routing are...
  evals           Memory evaluation commands.

Key subcommands you can run from the repo:

```bash
# Optimization proxy — sits between Claude Code and MiniMax.
uv run headroom proxy --port 8787

# Wrap Claude Code so settings.json points at the proxy automatically.
uv run headroom wrap claude -- --model MiniMax-M3

# Performance benchmark on local corpora.
uv run headroom perf scratch/

# Online evals suite.
uv run headroom evals

# Live traffic learning (extract error/recovery patterns from real requests).
uv run headroom proxy --memory --learn --port 8787
```

## 12. Where to go next

- **`notebooks/setting_up.ipynb`** — the deeper walkthrough: SmartCrusher + CacheAligner + MiniMax prompt cache, with paired baseline/compressed benchmarks.
- **`README.md`** — repo overview + reproduced 42% compression claim.
- **`CLAUDE.md`** — operational gotchas (`~/.claude/settings.json` `env` block, the LiteLLM `minimax/` pricing fix, etc.).
- **`scratch/tier1/`** — Tier-1 benchmarks (GSM8K, SQuAD, BFCL, TruthfulQA) using `headroom.compress()` in-process.
- **`scratch/langgraph/`** — LangGraph ReAct agent benchmark, MCP-backed tickets DB, ~42% measured compression on MiniMax-M3.
- **Upstream docs** — [headroom-docs.vercel.app](https://headroom-docs.vercel.app) (drifts; treat as starting point, verify against the installed package).